# Power Law RAES on threshold

In [2]:
using CSV, DataFrames

The Power Law RAES Model "on $c$" is a dynamic random graph generator based on the original RAES model. In this model, each node, at the beginning of the network formation process, assign itself a capacity threshold value $c_u$ according to the following rule:
- With probability $p,$ it sets its capacity threshold to a standard value $c,$ that is, $c_u = c.$
- With probability $1-p,$ it sets its capacity threshold according to a power law distribution with exponent $k,$ i.e, $c_u \sim \text{PowerLaw}(k).$

The model is defined by the following parameters:
- $n:$ number of nodes in the network.
- $d:$ minimum target degree.
- $c:$ standard capacity threshold multiplier, used to define the maximum degree tolerance.
- $p:$ Probability of standard behaviour adoption.
- $k:$ Power law exponent.

The graph-generating dynamic is defined as it follows:

- Initialization phase: 
- Starting from an arbitrary initial graph $G_0=(V,E_0)$
    - Each node $u \in V$
        - Set the threshold as the standard one, that is, $c_u = c$ with probability $p,$ otherwise
        - Set the threshold according to a power law distribution with exponent $k.$ 
- At each round $t \in \mathbb{N}$
    - Step $1:$ For each node $u \in V,$ let $N^1_u$ be the set of neighbors of $u$ at the beginning of Step $1.$ If $|N^1_u| < d$ then $u$ samples $d - |N^1_u|$ nodes from the set $V \setminus N^1_u,$ independently and u.a.r. with replacement and connects to them.
    - Step $2:$ For each node $u \in V,$ let $N^2_u$ be the set of neighbors of $u$ at the beginning of Step $2.$ If $|N^2_u| > c_u \cdot d,$ then $u$ samples $|N^2_u| - (c_u \cdot d)$ neighbors from the set $N^2_u,$ independently and u.a.r. with replacement, and disconnects from them.

## Experiments

### Influence of $p$ on spectral expansion 

In this initial experiment, we investigate the topological impact of the probability parameter $p$. This parameter controls the network's transition from a fully heterogeneous state, dominated by a power-law capacity distribution ($p=0$), to a fully homogeneous regime governed by the standard RAES mechanism ($p=1$). Throughout the simulations, the network size was fixed at $n = 2^{15}$ nodes, with $d = 8$, $c = 16$, and $k = 2.0$. We systematically varied $p$ from $0.0$ to $1.0$ in increments of $0.1$, conducting $100$ independent runs for each value and allowing the network to evolve for $100$ rounds. Finally, we measured the mean Spectral Gap of the final topologies to assess how capacity heterogeneity affects the overall robustness of the network.

In [3]:
df = CSV.read("../results/plaw_c_raes/spectral_gap_vs_p.csv", DataFrame)
df

Row,p_value,mean_gap,std_gap
,Float64,Float64,Float64
1,0.0,0.444513,0.000617291
2,0.1,0.451876,0.00060571
3,0.2,0.459365,0.000568013
4,0.3,0.466721,0.000492545
5,0.4,0.474211,0.000511876
6,0.5,0.481438,0.00054915
7,0.6,0.488774,0.000477172
8,0.7,0.495934,0.00044993
9,0.8,0.50291,0.00040314


In the regime of maximum heterogeneity ($p = 0.0$), the network records its minimum value of connectivity, with a mean gap of $\approx 0.444$. As the percentage of "Standard" nodes with a high fixed capacity ($c = 16$) increases, the gap grows linearly. The expansion peak ($\approx 0.516$) is reached at $p = 1.0$, which is when the network recovers the homogeneous behavior of the original RAES protocol. This discrepancy indicates that the introduction of asymmetric capacities causes a deterioration of the Random Expander properties.

Another observation derives from the analysis of the gap's standard deviation (std_gap) across the $100$ independent executions. In purely homogeneous networks ($p=1.0$), the variance is minimal ($\approx 0.00024$), indicating that the standard protocol generates highly deterministic and predictable topologies. Conversely, in the heterogeneous regime ($p=0.0$), the standard deviation is nearly tripled ($\approx 0.00061$). This demonstrates that heterogeneity introduces a topological instability, as the final topology becomes highly dependent on which specific nodes end up drawing the most limiting capacities.

### Average Degree

In this second experiment, we analyze the average degree of the network. The goal is to quantify the overall "density" of connections as we transition from a purely heterogeneous environment to a standard homogeneous one.
The experimental setup is analogous at the previous one:
- Network Size: $n=2^{15}$ (32,768 nodes).
- Parameters: Target outbound degree $d=8$, standard capacity factor $c=16$, and Power law parameter $k=2.0$.

$p$ varied from $0.0$ to $1.0$ in increments of $0.1$.

We again executed $100$ independent runs of $100$ rounds each. The degree of the network is computed at the end of each execution.


In [6]:
df = CSV.read("../results/plaw_c_raes/average_degrees.csv", DataFrame)
df

Row,p_value,average_degree,std_degree
,Float64,Float64,Float64
1,0.0,11.8518,2.98238
2,0.1,12.2057,3.06128
3,0.2,12.5776,3.12203
4,0.3,12.9599,3.16233
5,0.4,13.3625,3.18839
6,0.5,13.7707,3.1909
7,0.6,14.1997,3.17028
8,0.7,14.6352,3.12577
9,0.8,15.079,3.05794


We observe a significant and monotonic increase in connectivity as the network becomes more homogeneous. At $p=0.0$ the average degree is at its lowest, approximately $11.85$. At $p=1.0$, the average degree reaches its peak at approximately $15.88$.

The data shows that introducing Power-Law heterogeneity ($p \to 0$) leads to a sparser topology. Although the "super-nodes" in the Power-Law distribution have the capacity to accept hundreds of connections, the network average degree actually drops. This is due the fact that the Power law distribution generates a majority of nodes with very low capacity factors (near $1.0$). These nodes initiate their $8$ connections but prune any connections that exceed their limits.

### Degree distributions

In this last experiment, we analyze the equilibrium degree distributions of the network nodes. Again, we performed simulations on a network of $n=2^{15}$ nodes over 100 independent runs, each lasting 100 rounds. The probability parameter $p$ was systematically varied from $0.0$ to $1.0$ in increments of $0.1$. Throughout the process, the structural parameters were held constant at $d=8$ and $c=16$, with a Power law parameter of $k=2.0$. This allows us to observe how the shift from a heterogeneous capacity model to a homogeneous one affects the local connectivity of the peers.

In [4]:
df = CSV.read("../results/plaw_c_raes/degree_distributions.csv", DataFrame)
df

Row,degree,p_0.0,p_0.1,p_0.2,p_0.3,p_0.4,p_0.5,p_0.6,p_0.7,p_0.8,p_0.9,p_1.0
,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,8,0.148298,0.123258,0.0997809,0.077674,0.0581244,0.0413083,0.0266919,0.0155209,0.00735779,0.00224335,0.0003479


The degree distribution matrix reports the probability (i.e., the fraction of nodes) of observing a specific total degree $k$, ranging from $0$ to $35,$ where $35$ is the maximum degree observed. 

First of all, we see that, across all values of $p$, the probability for degrees $0$ through $7$ is exactly $0.0$. This confirms that the first step of the protocol is working correctly, as every node in the network successfully establishes its self-initiated target of at least $d=8$ connections. This ensures that no node remains isolated and that the network maintains a baseline level of density regardless of capacity constraints.

The data shows a significant shift in the distribution's of the most frequent degree as the network becomes more heterogeneous ($p \to 0$):
- For $p=1,$ the distribution is centered around a peak at degree 16 ($\approx 14\%$). In this state, nodes maintain their 8 self-initiated edges and roughly 8 additional edges established by their peers.
- For $p=0$ the peak shifts to a most frequent degree of $8$ ($\approx 14.8\%$).

In the Power-law model, many nodes draw a capacity factor $c \approx 1.0$. These nodes fulfill their own requirement of $8$ edges but have no room to accept further connections from others. During step 2, they  prune any additional edges to stay within their capacity limits. This creates a large number of nodes at the minimum degree, which explains the overall drop in the network's average degree.

Another observation is the absence of massive hubs, even when $p=0.0$. Although the Power law distribution allows for theoretical capacities reaching into the hundreds, the maximum degree recorded in the network is only $35$ (observed at $p=1.0$), while for $p=0.0$ the distribution effectively cuts off around $31$.